# Simulation and goodness of fit

Section 1.3 of the lecture notes, its last subsection, at the keyboard.  Two algorithms produce
paths of the two-type process: thinning (Algorithm 1.3.1), which works for any kernel, and the
exact scheme (Algorithm 1.3.2), which the exponential kernel allows.  Sections 1 to 4 write both
out on raw arrays, reproduce the package's simulators draw for draw, and run the exact scheme
above $\rho = 1$, which the package's specification refuses.

Sections 5 to 7 ask whether a path is a path of the model: one simulator against the other,
each path against its own compensator (Theorem 1.3.5), and what that second test leaves
untested.

In [ ]:
import timeit
from itertools import islice, takewhile

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

from unito26.lob import config
from unito26.lob.hawkes import (
    Clock, ExponentialHawkes, HawkesParams, OgataThinningHawkes, compensators_at_events,
    decayed_counts_at, intensities_at_events, mean_response_integral,
)
from unito26.lob.visualization import (
    INK, MUTED, PALETTE, exponential_qq_figure, monte_carlo_figure, use_template,
)

use_template()

PARAMS = config.asymmetric_pair_params()
MU_BAR = PARAMS.baseline.sum()
LAMBDA_STAR = PARAMS.stationary_intensity()

print("mu              ", PARAMS.baseline)
print("A               ", PARAMS.excitation.tolist())
print("beta            ", PARAMS.decay)
print("Gamma           ", PARAMS.branching_matrix.tolist())
print("rho             ", round(PARAMS.branching_ratio, 6))
print("lambda*         ", LAMBDA_STAR)
print("relaxation time ", PARAMS.relaxation_time(), "s")

The asymmetric pair serves throughout, except where another process is named.  Its types are 1
and 2, as in the notes; an array indexes them from 0, and everything printed counts from 1.

## 1. Thinning, written out

Algorithm 1.3.1 draws candidates at the rate of a bound, which starts at the post-jump value
$\lambda_{\mathfrak g}(T_{n-1}+)$.  At a candidate's time $t$ it keeps the candidate with
probability $\lambda_{\mathfrak g}(t)$ over the bound; otherwise it lowers the bound to
$\lambda_{\mathfrak g}(t)$ and draws again.  The bound is free because $A \geq 0$ makes
$\lambda_{\mathfrak g}$ non-increasing between events, and with the exponential kernel
$\lambda_{\mathfrak g} = \bar\mu + \mathbf 1^\top A Z$ is read off the state.

The loop below draws its uniforms in the order `OgataThinningHawkes` draws them, and keeps every
candidate: its time, the bound it was drawn at, the height that decided it, uniform under the
bound, and the decision.  The type is drawn on the pre-jump intensities by inverting their
cumulative sum, as in (1.3.40); section 3 uses the same function.

In [ ]:
def draw_type(baseline, excitation, state, rng):
    '''The type of an event at the current time, with probabilities lambda_e / lambda_g.'''
    cumulative = np.cumsum(baseline + excitation @ state)
    drawn = np.searchsorted(cumulative, rng.random() * cumulative[-1], side="left")
    return min(int(drawn), baseline.size - 1)


def thinning(baseline, excitation, decay, horizon, rng):
    '''Algorithm 1.3.1 on the state Z, keeping every candidate.

    Returns the events as (times, types), and the candidates up to the horizon as rows
    (time, bound, height, kept).
    '''
    mu_bar, jumps = baseline.sum(), excitation.sum(axis=0)
    time, state = 0.0, np.zeros(baseline.size)
    times, types, candidates = [], [], []
    while True:
        bound = mu_bar + jumps @ state            # lambda_g(T_{n-1}+), post-jump
        kept = False
        while not kept:
            wait = -np.log(rng.random()) / bound
            time += wait
            state *= np.exp(-decay * wait)
            intensity = mu_bar + jumps @ state
            height = rng.random() * bound
            kept = height <= intensity
            candidates.append((time, bound, height, kept))
            bound = intensity
        if time > horizon:
            candidates = [candidate for candidate in candidates if candidate[0] <= horizon]
            return np.array(times), np.array(types), np.array(candidates)
        event_type = draw_type(baseline, excitation, state, rng)
        state[event_type] += 1.0
        times.append(time)
        types.append(event_type)


times, types, candidates = thinning(
    PARAMS.baseline, PARAMS.excitation, PARAMS.decay, 2000.0, np.random.default_rng(3)
)
package = list(OgataThinningHawkes(PARAMS, rng=3).events(2000.0))

print(f"{times.size:,} events from {len(candidates):,} candidates, "
      f"{int(candidates[:, 3].sum()):,} of them kept")
print("types as the package's       :", np.array_equal(types, [e for _, e in package]))
print("largest difference in a time :", np.max(np.abs(times - [t for t, _ in package])))

The same uniforms, spent in the same order, give the same path to the last bit.  The class
discards the candidates it rejects; the loop keeps them, and the rest of this section reads them.

In [ ]:
SHORT = 6.0
times, types, candidates = thinning(
    PARAMS.baseline, PARAMS.excitation, PARAMS.decay, SHORT, np.random.default_rng(2)
)
clock = Clock(np.linspace(0.0, SHORT, 3000))
total = MU_BAR + decayed_counts_at(
    PARAMS.decay, times, types, np.zeros(2), clock
) @ PARAMS.excitation.sum(axis=0)
when, bound, height, kept = candidates.T
kept = kept.astype(bool)

print(f"{times.size} events, {np.count_nonzero(~kept)} candidates rejected")

figure = go.Figure([
    go.Scatter(x=np.r_[0.0, when], y=np.r_[bound, bound[-1]], mode="lines", line_shape="hv",
               name="bound", line=dict(color=MUTED, width=1)),
    go.Scatter(x=clock, y=total, mode="lines", name="total intensity",
               line=dict(color=PALETTE[0], width=2)),
    go.Scatter(x=when[kept], y=height[kept], mode="markers", name="kept",
               marker=dict(color=INK, size=8)),
    go.Scatter(x=when[~kept], y=height[~kept], mode="markers", name="rejected",
               marker=dict(color=MUTED, size=9, symbol="circle-open", line=dict(width=2))),
])
figure.update_xaxes(title_text="time (s)", range=[0.0, SHORT])
figure.update_yaxes(title_text="intensity (1/s)", rangemode="tozero")
figure.update_layout(title="Thinning: every candidate under the bound, kept when under the curve",
                     height=440, legend=dict(orientation="h", y=1.1, x=0))
figure

Every candidate lies under the bound, and the kept ones under $\lambda_{\mathfrak g}$.  After a
rejection the bound drops to $\lambda_{\mathfrak g}$ at the candidate, so across a quiet stretch
the steps descend towards the curve; at an event they rise to the post-jump value.  The
candidates between the curve and the bound are the work thinning does and discards.

How much of it there is depends on how far $\lambda_{\mathfrak g}$ decays during one wait.  A
candidate drawn when the excited part of $\lambda_{\mathfrak g}$ is $D$, so that the bound is
$\bar\mu + D$, arrives after $W \sim \mathrm{Exp}(\bar\mu + D)$ and is kept with probability
$(\bar\mu + De^{-\beta W})/(\bar\mu + D)$.  Since
$\mathbb E\,e^{-\beta W} = (\bar\mu + D)/(\bar\mu + D + \beta)$,

$$\mathbb P(\text{rejected} \mid D)
= \frac{D}{\bar\mu + D}\Big(1 - \mathbb E\,e^{-\beta W}\Big)
= \frac{D\beta}{(\bar\mu + D)(\bar\mu + D + \beta)} .$$

It vanishes as $D \to 0$, where there is no excitation to lose, and as $D/\beta \to \infty$,
where the next candidate comes before the excitation has had time to decay.  We bin the
candidates of a long run on $D$ and compare.

In [ ]:
def rejection_probability(excess, mu_bar, decay):
    return excess * decay / ((mu_bar + excess) * (mu_bar + excess + decay))


_, _, candidates = thinning(
    PARAMS.baseline, PARAMS.excitation, PARAMS.decay, 20_000.0, np.random.default_rng(5)
)
drawn = pd.DataFrame({
    "excess": candidates[:, 1] - MU_BAR,
    "rejected": candidates[:, 3] == 0,
})
drawn["formula"] = rejection_probability(drawn["excess"], MU_BAR, PARAMS.decay)
binned = drawn.groupby(pd.qcut(drawn["excess"], 20), observed=True).agg(
    excess=("excess", "mean"), rejected=("rejected", "mean"),
    formula=("formula", "mean"), candidates=("rejected", "size"),
)
binned["error"] = np.sqrt(binned["rejected"] * (1 - binned["rejected"]) / binned["candidates"])
departure = (binned["rejected"] - binned["formula"]).abs() / binned["error"]

print(f"{len(drawn):,} candidates in {len(binned)} bins of D")
print(f"largest |measured - formula| in a bin: {departure.max():.2f} standard errors; "
      f"bins beyond two: {np.count_nonzero(departure > 2)} of {len(binned)}")

grid = np.geomspace(binned["excess"].iloc[0], binned["excess"].iloc[-1], 200)
figure = go.Figure([
    go.Scatter(x=grid / PARAMS.decay, y=rejection_probability(grid, MU_BAR, PARAMS.decay),
               mode="lines", name="formula", line=dict(color=INK, width=2)),
    go.Scatter(x=binned["excess"] / PARAMS.decay, y=binned["rejected"], mode="markers",
               name="measured, ±2 SE", marker=dict(color=PALETTE[0], size=8),
               error_y=dict(type="data", array=2 * binned["error"], color=PALETTE[0])),
])
figure.update_xaxes(type="log", title_text="D / β")
figure.update_yaxes(title_text="share of candidates rejected", rangemode="tozero")
figure.update_layout(title="The share rejected, binned on the excited part D",
                     height=420, legend=dict(orientation="h", y=1.1, x=0))
figure

The measured share follows the formula across the range of $D$, and the few bins beyond two
standard errors are what twenty bins show by chance.

The formula involves $D$, $\bar\mu$ and $\beta$, and no candidate's fate changes when time is
rescaled: $(\mu, A, \beta) \mapsto (c\mu, cA, c\beta)$ divides every wait by $c$, multiplies
every rate by $c$, and leaves $\Gamma$ and the clustering number $\bar\lambda/\beta$ of
Definition 1.3.37 where they were (Remark 1.3.38).  At a fixed $\Gamma$ and a fixed composition of
$\lambda^*$, the share kept is therefore a function of $\bar\lambda/\beta$ alone.  The left
panel below reaches each clustering number twice: by varying $\beta$ at fixed $\lambda^*$, and by
scaling $\lambda^*$ at $\beta = 4$.

The right panel holds $\lambda^* = (1, 1)$ and $\beta = 4$, and with them $\bar\lambda/\beta$, and
moves $\rho$ along the shape of the self-exciting pair, whose ladder at fixed $\lambda^*$ reaches
$\rho \to 1$.  Each point is one run of about twenty thousand events.

In [ ]:
def candidates_of(params, events, seed):
    '''The candidates of a thinning run long enough for about `events` events.'''
    horizon = events / params.stationary_intensity().sum()
    rng = np.random.default_rng(seed)
    return thinning(params.baseline, params.excitation, params.decay, horizon, rng)[2]


def share_kept(candidates):
    return candidates[:, 3].mean()


DECAYS = PARAMS.decay * 2.0 ** np.arange(-4, 5)
by_decay = [
    HawkesParams(baseline=PARAMS.baseline, excitation=decay * PARAMS.branching_matrix, decay=decay)
    for decay in DECAYS
]
by_scale = [
    HawkesParams(baseline=PARAMS.baseline * PARAMS.decay / decay, excitation=PARAMS.excitation,
                 decay=PARAMS.decay)
    for decay in DECAYS
]
kept_by_decay = [share_kept(candidates_of(p, 20_000, 100 + k)) for k, p in enumerate(by_decay)]
kept_by_scale = [share_kept(candidates_of(p, 20_000, 200 + k)) for k, p in enumerate(by_scale)]
clustering = pd.DataFrame({
    "lambda_bar / beta": [p.stationary_intensity().sum() / p.decay for p in by_decay],
    "beta varied": kept_by_decay,
    "lambda* scaled": kept_by_scale,
})
print("share of candidates kept, asymmetric pair")
print(clustering.round(4).to_string(index=False))

SELF = config.self_exciting_pair_params()
RHOS = np.arange(1, 20) / 20
ladder = [
    HawkesParams.from_stationary_intensity(np.ones(2), SELF.excitation, SELF.decay, rho)
    for rho in RHOS
]
ladder_candidates = [candidates_of(p, 20_000, 300 + k) for k, p in enumerate(ladder)]
by_rho = pd.DataFrame({
    "rho": RHOS,
    "mu_bar": [p.baseline.sum() for p in ladder],
    "kept": [share_kept(c) for c in ladder_candidates],
    "median D": [np.median(c[:, 1] - p.baseline.sum()) for c, p in zip(ladder_candidates, ladder)],
})
print()
print("share of candidates kept, lambda* = (1, 1), beta = 4, self-exciting shape")
print(by_rho.round(4).to_string(index=False))

In [ ]:
figure = make_subplots(
    rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.06,
    subplot_titles=("against the clustering number, at ρ = 0.6", "against ρ, at λ̄/β = 0.5"),
)
routes = (("beta varied", "β varied at λ* = (2, 1)"), ("lambda* scaled", "λ* scaled at β = 4"))
for slot, (column, label) in enumerate(routes):
    figure.add_trace(go.Scatter(
        x=clustering["lambda_bar / beta"], y=clustering[column], mode="markers", name=label,
        marker=dict(color=PALETTE[slot], size=9 - 3 * slot),
    ), row=1, col=1)
figure.add_trace(go.Scatter(
    x=by_rho["rho"], y=by_rho["kept"], mode="lines+markers", name="ρ varied at λ* = (1, 1)",
    line=dict(color=PALETTE[2], width=2), marker=dict(size=7),
), row=1, col=2)
figure.update_xaxes(type="log", title_text="clustering number λ̄/β", row=1, col=1)
figure.update_xaxes(title_text="ρ", row=1, col=2)
figure.update_yaxes(title_text="share of candidates kept", row=1, col=1)
figure.update_layout(title="What thinning keeps", height=440,
                     legend=dict(orientation="h", y=1.18, x=0))
figure

Along either route the share kept falls as $\bar\lambda/\beta$ falls: the faster the excitation
decays against the rate of events, the further $\lambda_{\mathfrak g}$ falls during a wait, and the
more candidates land above the curve.  The two routes trace one curve.

At a fixed $\bar\lambda/\beta$, $\rho$ moves the share both ways.  Raising $\rho$ at fixed
$\lambda^*$ takes rate from $\bar\mu$ and gives it to the excitation, and the median candidate
moves from $D$ well below $\bar\mu$, where the formula rejects little, to bursts in which $D$ is
well above $\beta$, where it rejects little again.  The share rejected peaks in between, and it
is not monotone in $\rho$.

## 2. The recursion

Each evaluation of $\lambda_{\mathfrak g}$ in the loop read the state, not the history.  Written as
the sum (1.3.11) over the past, $Z(T_n-)$ costs a term for every earlier event, and a path of $n$
events costs $O(n^2)$.  The update (1.3.36) carries the state from one event to the next at
$O(1)$:

$$Z(T_{n+1}-) = \big(Z(T_n-) + \mathbf e_{E_n}\big)\,e^{-\beta(T_{n+1} - T_n)} .$$

In [ ]:
def states_by_sum(decay, times, types):
    '''Z(T_n-) at every event, by the sum (1.3.11) over the events before it.'''
    arrivals = np.eye(2)[types]
    states = np.empty((times.size, 2))
    for n, now in enumerate(times):
        states[n] = np.exp(-decay * (now - times[:n])) @ arrivals[:n]
    return states


def states_by_recursion(decay, times, types):
    '''Z(T_n-) at every event, by the update (1.3.36) from the event before it.'''
    state, previous = np.zeros(2), 0.0
    states = np.empty((times.size, 2))
    for n, (now, event_type) in enumerate(zip(times, types)):
        state = state * np.exp(-decay * (now - previous))
        states[n] = state                 # Z(T_n-): enters the intensity
        state[event_type] += 1.0          # Z(T_n+): carried forward
        previous = now
    return states


path = list(islice(ExponentialHawkes(PARAMS, rng=4).events(np.inf), 16_000))
path_times = np.array([time for time, _ in path])
path_types = np.array([event_type for _, event_type in path])


def seconds(route, n):
    return min(timeit.repeat(
        lambda: route(PARAMS.decay, path_times[:n], path_types[:n]), number=1, repeat=3))


timings = pd.DataFrame(
    [(n, seconds(states_by_sum, n), seconds(states_by_recursion, n))
     for n in (1_000, 2_000, 4_000, 8_000, 16_000)],
    columns=["events", "sum (s)", "recursion (s)"],
).set_index("events")
timings["sum growth"] = timings["sum (s)"] / timings["sum (s)"].shift()
timings["recursion growth"] = timings["recursion (s)"] / timings["recursion (s)"].shift()
timings["sum / recursion"] = timings["sum (s)"] / timings["recursion (s)"]
print(timings.round(4).to_string())

In [ ]:
by_sum = states_by_sum(PARAMS.decay, path_times, path_types)
by_recursion = states_by_recursion(PARAMS.decay, path_times, path_types)
scale = np.maximum(np.abs(by_sum), np.abs(by_recursion))
relative = np.abs(by_sum - by_recursion)[scale > 0] / scale[scale > 0]
replayed = intensities_at_events(PARAMS, path_times, path_types)

print("largest relative difference, sum against recursion :", relative.max())
print("largest difference from intensities_at_events       :",
      np.abs(PARAMS.baseline + by_recursion @ PARAMS.excitation.T - replayed).max())

The recursion's time doubles with each doubling of the path.  The sum's grows by a larger factor,
since each event sums over all the ones before it, and the gap between the two widens with every
row.  The two agree to rounding, and with the package's own replay of the intensity.

Remark 1.3.40 separates the two values the update produces.  The pre-jump $Z(T_n-)$ enters the
intensity at $T_n$, by predictability; the post-jump $Z(T_n+) = Z(T_n-) + \mathbf e_{E_n}$ is
carried forward.  In $\lambda_{\mathfrak g}$ they differ by the column sum
$(\mathbf 1^\top A)_{E_n}$, the jump an event of type $E_n$ gives the total intensity.

In [ ]:
jumps = PARAMS.excitation.sum(axis=0)
first = slice(0, 6)
before = MU_BAR + by_recursion[first] @ jumps
after = MU_BAR + (by_recursion[first] + np.eye(2)[path_types[first]]) @ jumps

print("column sums of A:", jumps)
print(pd.DataFrame({
    "time": path_times[first],
    "type": path_types[first] + 1,
    "lambda_g(T-)": before,
    "lambda_g(T+)": after,
    "jump": after - before,
}).round(4).to_string(index=False))

Each algorithm needs both values.  Thinning bounds with the post-jump value, since the pre-jump
value is smaller by the jump and does not dominate, and draws the type on the pre-jump
intensities.  The exact scheme reads $D$ post-jump and draws the type pre-jump.  The compensator
of section 6 reads the state pre-jump.

## 3. The exact scheme, written out

With a common decay, $\lambda_{\mathfrak g}(t) = \bar\mu + De^{-\beta(t - T_n)}$ between events
(1.3.37), where $D = \mathbf 1^\top A Z(T_n+)$ is the excited part just after the last event.  The
survival function of the next wait is then a product (1.3.38), and the wait is the smaller of two
independent draws (1.3.39): $\tau_1 = -\log U_1/\bar\mu$ for the baseline, and
$\tau_2 = -\beta^{-1}\log(1 + \beta\log U_2/D)$ for the excitation, set to $+\infty$ when $D = 0$
or the argument of the logarithm is not positive.  The excitation clock is defective: with
probability $e^{-D/\beta}$ it never fires (Remark 1.3.43).

Algorithm 1.3.2 on raw arrays follows.  It draws in the order `ExponentialHawkes` draws: two
uniforms for every wait, the second drawn even when $D = 0$ and not used, then one for the type.
It yields each event with the $D$ it was drawn from and both waits.

In [ ]:
def exact_scheme(baseline, excitation, decay, rng):
    '''Algorithm 1.3.2 on raw arrays, yielding (time, type, D, tau_1, tau_2) for each event.'''
    mu_bar, jumps = baseline.sum(), excitation.sum(axis=0)
    time, state = 0.0, np.zeros(baseline.size)
    while True:
        excess = jumps @ state                    # D, from the post-jump state
        u1, u2 = rng.random(2)
        tau1 = -np.log(u1) / mu_bar
        tau2 = np.inf
        if excess > 0:
            remaining = 1.0 + decay * np.log(u2) / excess
            if remaining > 0:
                tau2 = -np.log(remaining) / decay
        wait = min(tau1, tau2)
        time += wait
        state *= np.exp(-decay * wait)
        event_type = draw_type(baseline, excitation, state, rng)
        state[event_type] += 1.0
        yield time, event_type, excess, tau1, tau2

Two checks of the draws, on a run of $10^5$ events.  Binned on $D$, the share of waits with
$\tau_2 = +\infty$ against $e^{-D/\beta}$.  And the share of events fired by the baseline clock,
$\tau_1 < \tau_2$.  The baseline clock is redrawn after every event and is memoryless, so it fires
at the constant rate $\bar\mu$ while events arrive at the mean rate $\bar\lambda$: its share of
events is $\bar\mu/\bar\lambda$, the reciprocal of the mean cluster size, which is the budget
$\bar\mu\bar C = \bar\lambda$ of Remark 1.3.23 read off the simulator.

In [ ]:
run = np.array(list(islice(
    exact_scheme(PARAMS.baseline, PARAMS.excitation, PARAMS.decay, np.random.default_rng(6)),
    100_000,
)))
_, _, excess, tau1, tau2 = run.T

waits = pd.DataFrame({"excess": excess, "defective": np.isinf(tau2)})
waits["formula"] = np.exp(-waits["excess"] / PARAMS.decay)
binned = waits.groupby(pd.qcut(waits["excess"], 20), observed=True).agg(
    excess=("excess", "mean"), defective=("defective", "mean"),
    formula=("formula", "mean"), waits=("defective", "size"),
)
binned["error"] = np.sqrt(binned["defective"] * (1 - binned["defective"]) / binned["waits"])
worst = ((binned["defective"] - binned["formula"]).abs() / binned["error"]).max()
print(f"largest |measured - formula| in a bin of D: {worst:.2f} standard errors")

baseline_fired = tau1 < tau2
batches = [batch.mean() for batch in np.array_split(baseline_fired, 50)]
print(f"share of events fired by the baseline clock: {baseline_fired.mean():.4f} "
      f"± {np.std(batches, ddof=1) / np.sqrt(len(batches)):.4f} (50 batch means)")
print(f"mu_bar / lambda_bar                        : {MU_BAR / LAMBDA_STAR.sum():.4f}")

grid = np.linspace(0.0, binned["excess"].iloc[-1], 200)
figure = go.Figure([
    go.Scatter(x=grid / PARAMS.decay, y=np.exp(-grid / PARAMS.decay), mode="lines",
               name="e<sup>-D/β</sup>", line=dict(color=INK, width=2)),
    go.Scatter(x=binned["excess"] / PARAMS.decay, y=binned["defective"], mode="markers",
               name="measured, ±2 SE", marker=dict(color=PALETTE[0], size=8),
               error_y=dict(type="data", array=2 * binned["error"], color=PALETTE[0])),
])
figure.update_xaxes(title_text="D / β, the direct offspring the state still owes")
figure.update_yaxes(title_text="share of waits with τ<sub>2</sub> = ∞", rangemode="tozero")
figure.update_layout(title="The excitation clock is defective", height=420,
                     legend=dict(orientation="h", y=1.1, x=0))
figure

Both checks hold within their errors.  Remark 1.3.43 names the error they are there for: an
implementation that clips $\tau_2$ where it should be infinite biases the share of events the
baseline fires.

Now the loop against the package, on one seed.  The class forms $D$ as
$\lambda_{\mathfrak g} - \bar\mu$ and the loop as $\mathbf 1^\top A Z$, so the times can agree to
rounding rather than to the bit.

In [ ]:
inline = list(islice(
    exact_scheme(PARAMS.baseline, PARAMS.excitation, PARAMS.decay, np.random.default_rng(11)),
    10_000,
))
package = list(islice(ExponentialHawkes(PARAMS, rng=11).events(np.inf), 10_000))
inline_times = np.array([event[0] for event in inline])
package_times = np.array([time for time, _ in package])

assert np.array_equal([event[1] for event in inline], [e for _, e in package])
np.testing.assert_allclose(inline_times, package_times, rtol=1e-12)

print(f"first {len(inline):,} events: types equal, times equal to rtol 1e-12")
print("largest relative difference in a time:",
      np.max(np.abs(inline_times - package_times) / package_times))

`OgataThinningHawkes` spends its uniforms differently, two for every candidate and a varying
number of candidates for every event, so it can agree with the exact scheme only in law.  Section
5 compares them that way.

## 4. Beyond $\rho = 1$

`HawkesParams` specifies a stationary process, and refuses $\rho \geq 1$.

In [ ]:
BASELINE = np.array([0.25, 0.25])
DECAY = 4.0
UNIT_SHAPE = SELF.branching_matrix / SELF.branching_ratio

try:
    HawkesParams(baseline=BASELINE, excitation=DECAY * 1.1 * UNIT_SHAPE, decay=DECAY)
except ValueError as error:
    print("refused:", error)

print("column sums of the shape at rho = 1:", UNIT_SHAPE.sum(axis=0))

Algorithm 1.3.2 asks for less: $A \geq 0$, a common $\beta$ and $\bar\mu > 0$, and nothing of
$\rho$.  So the loop of section 3 runs where the class does not.  We take the shape of the
self-exciting pair scaled to $\rho = 0.9$, $1$ and $1.1$, with $\beta = 4$ and
$\mu = (0.25, 0.25)$, and 2000 paths each from $Z(0) = 0$ to 10 s.  Its columns sum to one value,
so $v = \mathbf 1$ is a left Perron vector, $v^\top\lambda = \lambda_{\mathfrak g}$, and (1.3.35)
gives the mean total intensity:

$$\mathbb E\,\lambda_{\mathfrak g}(t) = \bar\mu\,\frac{\rho e^{\beta(\rho - 1)t} - 1}{\rho - 1}
\quad (\rho \neq 1),
\qquad
\mathbb E\,\lambda_{\mathfrak g}(t) = \bar\mu\,(1 + \beta t) \quad (\rho = 1) .$$

Integrating once more gives the expected number of events on $[0, t]$.  A second route to the
mean needs no Perron vector: $\mathbb E\,\lambda(t) = \mu + A\Phi(t)\mu$, from (1.3.30) at $z = 0$,
with $\Phi$ computed at any $\rho$ by `mean_response_integral`.  Each path is capped at
$5 \times 10^4$ events, and the cap is asserted never reached.

In [ ]:
HORIZON = 10.0
CAP = 50_000
CLOCK = Clock(np.linspace(0.0, HORIZON, 51))
CRITICAL = (0.9, 1.0, 1.1)


def mean_total_intensity(rho, t):
    '''(1.3.35) with v = 1, from Z(0) = 0.'''
    if rho == 1.0:
        return BASELINE.sum() * (1 + DECAY * t)
    return BASELINE.sum() * (rho * np.exp(DECAY * (rho - 1) * t) - 1) / (rho - 1)


def expected_count(rho, t):
    '''The integral of mean_total_intensity over [0, t].'''
    if rho == 1.0:
        return BASELINE.sum() * (t + DECAY * t**2 / 2)
    growth = DECAY * (rho - 1)
    return BASELINE.sum() * (rho * (np.exp(growth * t) - 1) / growth - t) / (rho - 1)


def mean_total_by_phi(rho, t):
    '''1'(mu + A Phi(t) mu): the mean total intensity from (1.3.30) at z = 0.'''
    excitation = DECAY * rho * UNIT_SHAPE
    return (BASELINE + excitation @ mean_response_integral(excitation, DECAY, t) @ BASELINE).sum()


def supercritical_paths(rho, paths, seed):
    '''lambda_g on CLOCK and the number of events to HORIZON, one row per path.'''
    excitation = DECAY * rho * UNIT_SHAPE
    rng = np.random.default_rng(seed)
    totals, counts = np.empty((paths, CLOCK.size)), np.empty(paths)
    for k in range(paths):
        events = list(takewhile(
            lambda event: event[0] <= HORIZON,
            islice(exact_scheme(BASELINE, excitation, DECAY, rng), CAP),
        ))
        assert len(events) < CAP
        times = np.array([event[0] for event in events])
        types = np.array([event[1] for event in events], dtype=int)
        states = decayed_counts_at(DECAY, times, types, np.zeros(2), CLOCK)
        totals[k] = (BASELINE + states @ excitation.T).sum(axis=1)
        counts[k] = len(events)
    return totals, counts


runs = {rho: supercritical_paths(rho, 2000, seed) for rho, seed in zip(CRITICAL, (7, 8, 9))}

counted, read = [], []
for rho, (totals, counts) in runs.items():
    formula = mean_total_intensity(rho, CLOCK)
    error = totals.std(axis=0, ddof=1) / np.sqrt(counts.size)
    counted.append({
        "rho": rho,
        "E N(10 s)": expected_count(rho, HORIZON),
        "Monte Carlo": counts.mean(),
        "SE": counts.std(ddof=1) / np.sqrt(counts.size),
        "largest N(10 s) on a path": int(counts.max()),
    })
    read.append({
        "rho": rho,
        "E lambda_g(10 s)": formula[-1],
        "Monte Carlo": totals[:, -1].mean(),
        "SE": error[-1],
        "largest |z| on the clock": np.max(np.abs(totals.mean(axis=0) - formula)[1:] / error[1:]),
        "routes to the formula differ by": max(
            abs(mean_total_by_phi(rho, t) - mean_total_intensity(rho, t)) for t in CLOCK
        ),
    })
print(pd.DataFrame(counted).round(3).to_string(index=False))
print()
print(pd.DataFrame(read).to_string(
    index=False, float_format=lambda value: f"{value:.3g}" if abs(value) < 1e-3 else f"{value:.3f}"
))
print()
print(f"stationary total rate at rho = 0.9: {BASELINE.sum() / (1 - 0.9):.3f} per second")
print(f"doubling time at rho = 1.1       : {np.log(2) / (DECAY * (1.1 - 1)):.3f} s")

In [ ]:
figure = monte_carlo_figure(
    CLOCK,
    np.stack([runs[rho][0] for rho in CRITICAL], axis=-1),
    np.column_stack([mean_total_intensity(rho, CLOCK) for rho in CRITICAL]),
    [f"ρ = {rho}" for rho in CRITICAL],
    "Mean total intensity from Z(0) = 0: Monte Carlo, ±2 SE, and (1.3.35)",
)
figure.update_xaxes(title_text="time (s)")
figure.update_yaxes(type="log", title_text="mean total intensity (1/s), log scale")
figure

The mean is taken over the paths first and the logarithm after, because (1.3.35) gives
$\mathbb E\,\lambda_{\mathfrak g}$, and the mean of $\log\lambda_{\mathfrak g}$ is a different
number.  Below 1 the mean saturates at the stationary rate; at 1 it grows linearly; above 1 it
grows exponentially at the rate $\beta(\rho - 1)$, a straight line on the log axis, doubling every
$\log 2/(\beta(\rho - 1))$.

The Monte Carlo means follow the formula at all three values of $\rho$, and the two routes to the
formula agree to rounding.  The band is pointwise and each curve is read at fifty times, so its
largest departure is the largest of fifty, and at two or three standard errors it is no evidence
against the formula.  The times of one curve share their paths, so a curve errs in runs rather
than time by time.

Every path reached 10 s with a finite number of events, well short of the cap, and the expected
count is finite on every horizon, at every $\rho$ (Remark 1.3.14).  A linear Hawkes process does
not blow up in finite time.  Supercritical means a mean that grows exponentially; it does not
mean infinitely many events in a finite time.

## 5. One simulator tests the other

`ExponentialHawkes` and `OgataThinningHawkes` share the state, its bookkeeping and the type draw,
all in their common base class, and differ only in `step`, which draws the wait.  Comparing them
tests the wait, and nothing they share.

They consume their uniforms differently, so they are compared in law: one number per path, over
2000 independent paths per simulator, from empty, to 20 relaxation times.  The statistics are the
counts $N_1(T)$ and $N_2(T)$, the number of type switches, and the wait from the first event to
the second.  From empty the first event comes after an $\mathrm{Exp}(\bar\mu)$ wait under both
schemes by construction; the wait after it is the first one the excitation shortens.

In [ ]:
HORIZON_5 = 20 * PARAMS.relaxation_time()


def path_statistics(simulator_class, paths, seed):
    '''One row of statistics per independent path, and the gaps of all paths pooled.'''
    rng = np.random.default_rng(seed)
    rows, gaps = [], []
    for _ in range(paths):
        path = list(simulator_class(PARAMS, rng=rng).events(HORIZON_5))
        times = np.array([time for time, _ in path])
        types = np.array([event_type for _, event_type in path])
        assert times.size >= 2
        counts = np.bincount(types, minlength=2)
        rows.append((counts[0], counts[1], np.count_nonzero(np.diff(types)), times[1] - times[0]))
        gaps.append(np.diff(times))
    columns = ["N_1(T)", "N_2(T)", "type switches", "T_2 - T_1"]
    return pd.DataFrame(rows, columns=columns), np.concatenate(gaps)


exact, exact_gaps = path_statistics(ExponentialHawkes, 2000, 1)
thinned, thinned_gaps = path_statistics(OgataThinningHawkes, 2000, 2)

print(f"T = {HORIZON_5} s, 2000 paths per simulator")
print(pd.DataFrame({
    "exact scheme": exact.mean(),
    "thinning": thinned.mean(),
    "two-sample KS p-value": [stats.ks_2samp(exact[c], thinned[c]).pvalue for c in exact.columns],
}).round(4).to_string())

No p-value is small: none of the four statistics tells the two simulators apart.  On the
integer-valued statistics the two-sample Kolmogorov–Smirnov test is conservative: ties make its
p-values too large rather than too small.

The pooled gaps are drawn below for inspection, with no p-value.  Gaps along one path are
serially dependent, so the pooled gaps are not independent draws, and a two-sample test that
treats them as such overstates its own evidence.

In [ ]:
levels = np.linspace(0.001, 0.999, 400)
exact_quantiles = np.quantile(exact_gaps, levels)
figure = go.Figure([
    go.Scatter(x=exact_quantiles, y=exact_quantiles, mode="lines", hoverinfo="skip",
               line=dict(color=INK, width=1, dash="dot")),
    go.Scatter(x=exact_quantiles, y=np.quantile(thinned_gaps, levels), mode="markers",
               marker=dict(color=PALETTE[0], size=5),
               hovertemplate="exact %{x:.4f} s<br>thinning %{y:.4f} s<extra></extra>"),
])
figure.update_xaxes(type="log", title_text="gap, exact scheme (s)")
figure.update_yaxes(type="log", title_text="gap, thinning (s)")
figure.update_layout(title="Pooled gaps, quantile against quantile, for inspection only",
                     height=460, showlegend=False)
figure

The comparison cannot see what the two share.  A wrong type draw, or a state updated with
the wrong jump, would be wrong in both, identically, and every statistic above would agree as well
as it does here.  The type draw is tested in the next section, by a compensator that shares
nothing with either simulator.

## 6. Meyer's time change

Theorem 1.3.5: if the path is a path of the model, each type's rescaled inter-arrival times
$\tau^e_j = \Lambda_e(T^e_j) - \Lambda_e(T^e_{j-1})$ (1.3.7) are independent $\mathrm{Exp}(1)$
variables.  Its hypotheses are no common jumps, a continuous compensator and a divergent one.  It
needs neither stationarity nor $\rho < 1$, so the path starts from empty and nothing is discarded.

The compensator is (1.3.15), with the count read before $T$ and the state at it (Remark 1.3.12).
`compensators_at_events` evaluates it at every event by its own replay of the path, and shares no
code with either simulator.  Agreement is then evidence, where a compensator accumulated inside
the simulator would agree with the path by construction.

In [ ]:
LONG = 1e5
long_path = list(ExponentialHawkes(PARAMS, rng=2026).events(LONG))
long_times = np.array([time for time, _ in long_path])
long_types = np.array([event_type for _, event_type in long_path])
replayed = compensators_at_events(PARAMS, long_times, long_types)


def residuals(compensator_at_events, types):
    '''(1.3.7) for each type: the increments of its own compensator between its arrivals.'''
    return [np.diff(compensator_at_events[types == e, e], prepend=0.0) for e in range(2)]


def kolmogorov_smirnov(sample):
    '''sqrt(n) D_n against Exp(1), and its p-value.'''
    test = stats.kstest(sample, "expon")
    return np.sqrt(sample.size) * test.statistic, test.pvalue


rescaled = residuals(replayed, long_types)

print(f"{long_times.size:,} events on [0, {LONG:,.0f}] s; coincident arrival times: "
      f"{np.count_nonzero(np.diff(long_times) == 0)}")
print(f"5% critical value of sqrt(n) D_n: {stats.kstwobign.ppf(0.95):.3f}")
for e, sample in enumerate(rescaled, start=1):
    root_n_d, p = kolmogorov_smirnov(sample)
    print(f"type {e}: {sample.size:>7,} residuals, mean {sample.mean():.4f}, "
          f"sqrt(n) D_n = {root_n_d:.3f}, p = {p:.3f}")

In [ ]:
exponential_qq_figure(
    {"type 1": rescaled[0], "type 2": rescaled[1]}, 1000,
    "Rescaled inter-arrival times against Exp(1): pointwise band (dark), simultaneous (light)",
)

Both types follow the diagonal, and both statistics fall below the 5% critical value, which
places every point of either plot inside the simultaneous band.

Three wrong compensators, read on the same path, serve as controls: the transposed kernel $A^\top$,
which passes every stability check since $\rho(\Gamma^\top) = \rho(\Gamma)$; and the right $A$
with a wrong decay, $\beta/2$ and $1.25\beta$.  `HawkesParams` refuses $A$ at $\beta/2$, where the
branching ratio exceeds 1, so the controls use (1.3.15) written on raw arrays, on any clock, with
the state from `decayed_counts_at`.  On the model it must agree with `compensators_at_events`.

A compensator built from $(A', \beta')$ grows along the path at the rate $\mu + A'Z'$, with $Z'$
the state decayed at $\beta'$.  The time average of $Z'$ is $\lambda^*/\beta'$ at any decay, by
the identity in the proof of Proposition 1.3.11, so the residuals of type $e$ have mean
$(\mu + A'\lambda^*/\beta')_e/\lambda^*_e$.  On the model that is 1, since
$\mu + \Gamma\lambda^* = \lambda^*$.

In [ ]:
def compensator(baseline, excitation, decay, times, types, clock):
    '''(1.3.15) on a clock: the count before each time of the clock, and the state at it.'''
    counts = np.column_stack(
        [np.searchsorted(times[types == e], clock, side="left") for e in range(baseline.size)]
    )
    states = decayed_counts_at(decay, times, types, np.zeros(baseline.size), clock)
    return np.outer(clock, baseline) + (counts - states) @ excitation.T / decay


at_events = Clock(long_times)
closed_form = compensator(
    PARAMS.baseline, PARAMS.excitation, PARAMS.decay, long_times, long_types, at_events
)
print("branching ratio of A at beta / 2:",
      np.max(np.abs(np.linalg.eigvals(PARAMS.excitation / (PARAMS.decay / 2)))))
print("largest relative difference from compensators_at_events:",
      np.max(np.abs(closed_form - replayed) / replayed))

READINGS = {
    "the model": (PARAMS.excitation, PARAMS.decay),
    "transposed kernel": (PARAMS.excitation.T, PARAMS.decay),
    "decay β/2": (PARAMS.excitation, PARAMS.decay / 2),
    "decay 1.25β": (PARAMS.excitation, 1.25 * PARAMS.decay),
}
controls, rows = {}, []
for name, (excitation, decay) in READINGS.items():
    controls[name] = residuals(
        compensator(PARAMS.baseline, excitation, decay, long_times, long_types, at_events),
        long_types,
    )
    predicted = (PARAMS.baseline + excitation @ LAMBDA_STAR / decay) / LAMBDA_STAR
    for e, sample in enumerate(controls[name]):
        rows.append({
            "compensator": name, "type": e + 1, "mean": sample.mean(),
            "predicted mean": predicted[e], "sqrt(n) D_n": kolmogorov_smirnov(sample)[0],
        })
print()
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
levels = np.linspace(0.0025, 0.9975, 400)
reference = stats.expon.ppf(levels)
figure = make_subplots(rows=1, cols=2, horizontal_spacing=0.08)
for column in (1, 2):
    figure.add_trace(go.Scatter(
        x=reference, y=reference, mode="lines", showlegend=False, hoverinfo="skip",
        line=dict(color=INK, width=1, dash="dot"),
    ), row=1, col=column)
    for slot, (name, samples) in enumerate(controls.items()):
        figure.add_trace(go.Scatter(
            x=reference, y=np.quantile(samples[column - 1], levels), mode="lines", name=name,
            legendgroup=name, showlegend=column == 1, line=dict(color=PALETTE[slot], width=2),
        ), row=1, col=column)
    figure.update_xaxes(title_text=f"Exp(1) quantile, type {column}", row=1, col=column)
figure.update_yaxes(title_text="quantile of the rescaled times", row=1, col=1)
figure.update_layout(title="The same path through four compensators", height=440,
                     legend=dict(orientation="h", y=1.1, x=0))
figure

Every wrong compensator moves the quantiles off the diagonal, type 2 at $1.25\beta$ least visibly,
and every one of their statistics lies far beyond the critical value.  The measured means land
on the predicted ones.

The post-jump reading of Remark 1.3.12, $\mu T + \Gamma\big(N(T) - Z(T)\big)$, counts the event
at $T$ itself: `side="right"` where `compensator` has `side="left"`.  At an arrival of type $e$ it
overstates $\Lambda_e$ by $\Gamma_{ee}$.

In [ ]:
counts_through = np.column_stack(
    [np.searchsorted(long_times[long_types == e], long_times, side="right") for e in range(2)]
)
states = decayed_counts_at(PARAMS.decay, long_times, long_types, np.zeros(2), at_events)
post_jump = residuals(
    np.outer(long_times, PARAMS.baseline)
    + (counts_through - states) @ PARAMS.branching_matrix.T,
    long_types,
)
pre_jump = residuals(closed_form, long_types)

for e in range(2):
    moved = np.flatnonzero(~np.isclose(post_jump[e], pre_jump[e], rtol=0.0, atol=1e-9))
    print(f"type {e + 1}: {moved.size} of {pre_jump[e].size:,} residuals moved, at positions "
          f"{moved.tolist()}, by {post_jump[e][moved[0]] - pre_jump[e][moved[0]]:.6f}; "
          f"Gamma_ee = {PARAMS.branching_matrix[e, e]}")
    print(f"        sqrt(n) D_n = {kolmogorov_smirnov(pre_jump[e])[0]:.4f} pre-jump, "
          f"{kolmogorov_smirnov(post_jump[e])[0]:.4f} post-jump")

One residual per type moves, the first, and by $\Gamma_{ee}$.  Every later residual is an increment
between two arrivals of its type, and the constant added at both ends cancels.  The per-type test
cannot see this error.

## 7. What the test does not see

Section 6 compared each type's residuals with $\mathrm{Exp}(1)$, one type at a time.
Theorem 1.3.5 asserts more: the residuals are independent within a type, and the types are
independent of one another.  By Remark 1.3.45 a comparison of margins tests neither.

Within a type, the autocorrelation $r_k$ of the residuals at lags 1 to 20, against the band
$\pm 1.96/\sqrt n$ of an independent sample.  The band is pointwise, so the twenty lags are also
read at once, by the Ljung–Box statistic $n(n+2)\sum_k r_k^2/(n-k)$, which is $\chi^2_{20}$ for an
independent sample.

In [ ]:
LAGS = np.arange(1, 21)


def autocorrelation(sample, lags):
    centred = sample - sample.mean()
    return np.array([centred[:-lag] @ centred[lag:] for lag in lags]) / (centred @ centred)


def ljung_box(sample, correlations, lags):
    '''The Ljung-Box statistic over the given lags, and its p-value against chi-square.'''
    n = sample.size
    statistic = n * (n + 2) * np.sum(correlations**2 / (n - lags))
    return statistic, stats.chi2.sf(statistic, lags.size)


figure = make_subplots(rows=1, cols=2, subplot_titles=("type 1", "type 2"), shared_yaxes=True)
for e, sample in enumerate(rescaled):
    band = 1.96 / np.sqrt(sample.size)
    correlations = autocorrelation(sample, LAGS)
    statistic, p = ljung_box(sample, correlations, LAGS)
    print(f"type {e + 1}: band ±{band:.4f}, largest |autocorrelation| "
          f"{np.abs(correlations).max():.4f}, lags outside the band: "
          f"{LAGS[np.abs(correlations) > band].tolist()}; Ljung-Box {statistic:.1f} on "
          f"{LAGS.size} lags, p = {p:.3f}")
    figure.add_trace(go.Bar(
        x=LAGS, y=correlations, width=0.4, marker_color=PALETTE[e], showlegend=False,
    ), row=1, col=e + 1)
    figure.add_hrect(y0=-band, y1=band, fillcolor=MUTED, opacity=0.25, line_width=0,
                     layer="below", row=1, col=e + 1)
    figure.update_xaxes(title_text="lag", row=1, col=e + 1)
figure.update_yaxes(title_text="autocorrelation", row=1, col=1)
figure.update_layout(
    title="Autocorrelation of the rescaled times, and the band of an independent sample",
    height=380,
)
figure

An independent sample crosses a pointwise 95% band at about one lag in twenty, so a crossing or
two is not evidence of dependence; the joint statistic, which reads all twenty lags, finds none.

The same residuals as consecutive pairs, carried to the unit square by $U = 1 - e^{-\tau}$: under
the theorem the pairs $(U_j, U_{j+1})$ are uniform on the square.  Beside them, a control: the same
residuals, sorted within consecutive blocks of ten.  Sorting inside the blocks changes the order
and not the values, so it leaves the margin untouched, and with it the Kolmogorov–Smirnov
statistic, which depends on the sample only through its sorted values.

In [ ]:
first_type = rescaled[0][: rescaled[0].size // 10 * 10]
orders = {
    "as simulated": first_type,
    "sorted within blocks of ten": np.sort(first_type.reshape(-1, 10), axis=1).ravel(),
}
for name, sample in orders.items():
    root_n_d, p = kolmogorov_smirnov(sample)
    print(f"type 1, {name:<28}: sqrt(n) D_n = {root_n_d:.6f}, p = {p:.3f}, "
          f"lag-1 autocorrelation {autocorrelation(sample, [1])[0]:+.4f}")

figure = make_subplots(rows=1, cols=2, subplot_titles=list(orders), horizontal_spacing=0.08)
for column, sample in enumerate(orders.values(), start=1):
    uniform = 1 - np.exp(-sample[:3001])
    figure.add_trace(go.Scatter(
        x=uniform[:-1], y=uniform[1:], mode="markers", showlegend=False, hoverinfo="skip",
        marker=dict(color=PALETTE[0], size=3, opacity=0.5),
    ), row=1, col=column)
    figure.update_xaxes(title_text="U<sub>j</sub>", range=[0, 1], row=1, col=column)
    figure.update_yaxes(range=[0, 1], row=1, col=column)
figure.update_yaxes(title_text="U<sub>j+1</sub>", row=1, col=1)
figure.update_layout(title="Consecutive rescaled times of type 1 on the unit square",
                     height=460)
figure

The statistic is the same to every digit, and the square is not: the sorted residuals crowd
above the diagonal, and their autocorrelation at lag 1 lies far outside any band.  A test of the
margin reads only the sorted sample, so it cannot see the order.

Across types, the independence has a second-order trace that can be measured on a clock.
$M_e = N_e - \Lambda_e$ is a martingale for each type, and $\langle M_1, M_2\rangle \equiv 0$
(§1.3.4) because no two types jump together (1.3.1).  So over disjoint windows the increments
$\Delta M_1$ and $\Delta M_2$ are uncorrelated, although the counts $\Delta N_1$ and $\Delta N_2$
are not.  The products $\Delta M_1\Delta M_2$ are themselves martingale differences, which gives
the standard error of their correlation.

In [ ]:
def correlation_with_error(first, second):
    '''The correlation of two series of martingale increments, and its standard error.'''
    a, b = first - first.mean(), second - second.mean()
    norm = np.sqrt((a @ a) * (b @ b))
    return (a @ b) / norm, np.sqrt(np.sum((a * b) ** 2)) / norm


windows = Clock(np.arange(0.0, LONG + 1.0, 1.0))
counts = np.column_stack(
    [np.searchsorted(long_times[long_types == e], windows, side="right") for e in range(2)]
)
counts_increments = np.diff(counts, axis=0)
martingale_increments = counts_increments - np.diff(compensator(
    PARAMS.baseline, PARAMS.excitation, PARAMS.decay, long_times, long_types, windows
), axis=0)

print(f"{counts_increments.shape[0]:,} windows of 1 s")
print(f"correlation of the counts                : "
      f"{np.corrcoef(counts_increments.T)[0, 1]:+.4f}")
r, error = correlation_with_error(martingale_increments[:, 0], martingale_increments[:, 1])
print(f"correlation of the martingale increments : {r:+.4f} ± {error:.4f} (one SE)")

The counts are strongly correlated, since each type excites the other, and the martingale
increments are not: the compensator takes out everything the past predicts, and what is left of
the two types is orthogonal.

The control breaks the hypothesis this rests on.  One Poisson stream of rate 1, labelled as both
types, makes every event a common jump.  Each type on its own is a unit-rate Poisson process, its
compensator is $t$, and its margin passes.

In [ ]:
stream = np.random.default_rng(12).exponential(1.0, size=int(1.1 * LONG)).cumsum()
stream = stream[stream <= LONG]
root_n_d, p = kolmogorov_smirnov(np.diff(stream, prepend=0.0))
print(f"each type: the same {stream.size:,} residuals, sqrt(n) D_n = {root_n_d:.3f}, p = {p:.3f}")

common = np.diff(np.searchsorted(stream, windows, side="right")) - np.diff(windows)
print(f"correlation of the martingale increments: "
      f"{correlation_with_error(common, common)[0]:.4f}")

Both margins pass, and the two martingales have the same increments.  That is the no-common-jumps
failure of recorded data, where one aggressive order is written as several fills sharing a
timestamp.  A suite of per-type tests that passes has established the margins, and each of the
two independences needs a test of its own.